# Pre-body

## Clearing past runs (optional)

In [33]:
!rm -rf logs/ # clear logs
!rm -rf optimizer_output/

## IIC-OSIC Env Setup

In [34]:
# # The following workaround is needed to run the jupyter notebook in docker container.
import os
import subprocess

# Run the script in a shell, capture its environment
PDK = "ihp-sg13g2"
command = f"bash -c 'source /foss/tools/sak/iic-pdk-script.sh {PDK} && source ~/.bashrc && env'"
result = subprocess.run(command, capture_output=True, text=True, shell=True)

# Parse environment variables from the output
for line in result.stdout.splitlines():
    key, _, value = line.partition("=")
    if key and value:
        os.environ[key] = value
        
os.environ["PATH"] += ":/foss/tools/bin"

# Now they are in your current Python process
print("PDK_ROOT:", os.environ.get("PDK_ROOT"))
print("SPICE_USERINIT_DIR:", os.environ.get("SPICE_USERINIT_DIR"))

# Test ngspice
!ngspice -v

PDK_ROOT: /foss/pdks
SPICE_USERINIT_DIR: /foss/pdks/ihp-sg13g2/libs.tech/ngspice
******
** ngspice-44.2 : Circuit level simulation program
** Compiled with KLU Direct Linear Solver
** The U. C. Berkeley CAD Group
** Copyright 1985-1994, Regents of the University of California.
** Copyright 2001-2024, The ngspice team.
** Please get your ngspice manual from https://ngspice.sourceforge.io/docs.html
** Please file your bug-reports at http://ngspice.sourceforge.net/bugrep.html
** Creation Date: Sat May 24 09:38:33 UTC 2025
******


## Library Imports

In [35]:
import logging

from pathlib import Path

from symxplorer.spice_engine                import Spicelib_Wrapper, Sim_Execution_Type
from symxplorer.designer_tools              import Nevergrad_Spice_Multi_Spec_Constraint_Satisfaction, Project_Setup
from symxplorer.logging                     import setup_loggers

logger = logging.getLogger("SymXplorer.jupyter")
logger.info("Spicelib_Wrapper imported successfully.")

00:35:04 - SymXplorer.jupyter: [INFO] Spicelib_Wrapper imported successfully.


# Instantiations


## Loading the project config

In [36]:
# ----------------------------
# Instantiations
# ----------------------------
project_setup_yaml = Path(f"/foss/designs/eda/SymXplorer/examples/tunable-tia/ihp-sg13g2/spice/project_setup.yaml")
_ = setup_loggers()

# (1) Load the project setup information
PROJECT_SETUP = Project_Setup.from_yaml(project_setup_yaml)
PROJECT_SETUP

00:35:04 - SymXplorer: [INFO] 🚀 Logger initialized and ready!
00:35:04 - SymXplorer: [INFO] 📄 Log file: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/logs/SymXplorer_2025-10-02_00-35-04.log
00:35:04 - SymXplorer: [INFO] 🔧 spicelib logger set to 50
00:35:04 - SymXplorer.domains: [INFO] 📂 Loading project setup from /foss/designs/eda/SymXplorer/examples/tunable-tia/ihp-sg13g2/spice/project_setup.yaml
00:35:04 - SymXplorer.domains: [INFO] Initialized OptimizerConfig: CMA, type=nevergrad, budget=150, random_seed=48
00:35:04 - SymXplorer.domains: [INFO] 	Linear bounds: min=0, max=100
00:35:04 - SymXplorer.domains: [INFO] 	Log bounds: min=1, max=100
00:35:04 - SymXplorer.domains: [INFO] 	Loss function: max_loss=inf, norm_method=min-max, type=mse, rescale_mag=True, include_phase_loss=False, include_mag_loss=True
00:35:04 - SymXplorer.domains: [INFO] 	Number of target specs: 3
00:35:04 - SymXplorer.domains: [INFO] 		- TargetSpec(name=fc, target=100e6, range=1.00e+07 tolerance=500000

Project_Setup(name='Tunable-TIA', description='Tunable TIA BPF example sizing in the ihp-sg13g2 technology', simulator='ngspice', ws_root=PosixPath('/foss/designs/eda/SymXplorer'), netlist=PosixPath('examples/tunable-tia/ihp-sg13g2/spice/tb_ac.spice'), outdir=PosixPath('examples/tunable-tia/scripts/optimizer_output'), tech_spec=TechSpec(name='ihp-sg13g2', constraints={'max_nfet_w': np.float64(9.999999999999999e-06), 'min_nfet_w': np.float64(1.8e-07), 'max_nfet_l': np.float64(9.999999999999999e-06), 'min_nfet_l': np.float64(1.8e-07), 'max_pfet_w': np.float64(9.999999999999999e-06), 'min_pfet_w': np.float64(1.8e-07), 'max_pfet_l': np.float64(9.999999999999999e-06), 'min_pfet_l': np.float64(1.8e-07), 'max_cap_w': np.float64(0.01), 'min_cap_w': np.float64(1e-06), 'max_cap_l': np.float64(0.01), 'min_cap_l': np.float64(1e-06), 'max_res_w': np.float64(0.001), 'min_res_w': np.float64(1e-06), 'max_res_l': np.float64(0.001), 'min_res_l': np.float64(1e-06)}), pvt=PVT(temp=25, corner='tt', supply=

## Create a SPICE simulator wrapper

In [37]:
# (2) Create the Spice Simulator Wrapper
netlist_filename = Path(PROJECT_SETUP.ws_root) / Path(PROJECT_SETUP.netlist)
output_folder    = Path(PROJECT_SETUP.ws_root) / Path(PROJECT_SETUP.outdir)

wrapper = Spicelib_Wrapper(
    project_name=PROJECT_SETUP.name,
    netlist_filename=netlist_filename,
    output_folder=output_folder,
    sim_execution_t=Sim_Execution_Type.RUN_AND_WAIT,  # only RUN_AND_WAIT is supported as of now...,
    path_to_simulator=Path("/foss/tools/bin/ngspice"),
    verbose=False
    )
wrapper

00:35:04 - SymXplorer.spicelib: [INFO] 📂 Creating output directory for the first time: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/optimizer_output
00:35:04 - SymXplorer.spicelib: [INFO] --------------------------------------------------
00:35:04 - SymXplorer.spicelib: [INFO] 🚀 Spicelib_Wrapper initialized successfully!
00:35:04 - SymXplorer.spicelib: [INFO] 	📝 Project: Tunable-TIA
00:35:04 - SymXplorer.spicelib: [INFO] 	📜 Schematic: tb_ac
00:35:04 - SymXplorer.spicelib: [INFO] 	📂 Output Folder: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/optimizer_output
00:35:04 - SymXplorer.spicelib: [INFO] --------------------------------------------------
00:35:04 - SymXplorer.spicelib: [INFO] Using ngspice from ['/foss/tools/bin/ngspice']
00:35:04 - SymXplorer.spicelib: [INFO] 📊 --- Circuit Information ---
00:35:04 - SymXplorer.spicelib: [INFO] 🔗 Nodes in the netlist: ['VSS', 'GND', 'VDD', 'Vbias', 'Von', 'Vop', 'In', 'Ip']
00:35:04 - SymXplorer.spicelib: [INFO] Testbe

## Create an optimizer object

In [38]:
circuit_optimizer = Nevergrad_Spice_Multi_Spec_Constraint_Satisfaction(
    spicelib_wrapper=wrapper,
    setup_obj=PROJECT_SETUP
)
circuit_optimizer

00:35:04 - SymXplorer.optimizer: [INFO] Initialized the Nevergrad_Spice_Multi_Spec_Optimizer with 3 target specs


## Sanity Check

In [39]:
# wrapper.run_sanity_check(
#     use_editor=True,
#     sim_execution_t=Sim_Execution_Type.RUN_NOW
# )

# Main Body

## Optimization

In [40]:
circuit_optimizer.parameterize()

Dict(vbias=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_cap_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_cap_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_nfet_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_nfet_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_res_3_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_res_3_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_res_s_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_res_s_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}]):{'x_dut_nfet_w': 50.0, 'x_dut_nfet_l': 50.0, 'x_dut_cap_w': 50.0, 'x_dut_cap_l': 50.0, 'x_dut_res_s_l': 50.0, 'x_dut_res_s_w': 50.0, 'x_dut_res_3_l': 50.0, 'x_dut_res_3_w': 50.0, 'vbias': 50.0}

In [41]:
circuit_optimizer.optimize()

00:35:05 - SymXplorer.optimizer: [INFO] Optimization process started.
00:35:05 - SymXplorer.optimizer: [INFO] Optimizer is set to CMA with budget = 150
Optimizing: 100%|██████████| 150/150 [01:45<00:00,  1.42trial/s]
00:36:51 - SymXplorer.optimizer: [INFO] Optimization process completed.


[{'params': {'x_dut_nfet_w': 51.772167102342905,
   'x_dut_nfet_l': 52.56700864888934,
   'x_dut_cap_w': 59.78125738724318,
   'x_dut_cap_l': 48.566113170842435,
   'x_dut_res_s_l': 32.67877083350588,
   'x_dut_res_s_w': 40.45237352750712,
   'x_dut_res_3_l': 45.398649124283864,
   'x_dut_res_3_w': 47.56010978912369,
   'vbias': 40.79555921648511},
  'loss': np.float64(1.9246636224398537),
  'metadata': {'fc': {'curr_val': np.float64(7638218.0),
    'loss': np.float64(0.9998051203684322)},
   'q': {'curr_val': np.float64(8.624050455690952), 'loss': np.float64(0.0)},
   'gain_db': {'curr_val': np.float64(-24.864695269738707),
    'loss': np.float64(0.9248585020714215)}}},
 {'params': {'x_dut_nfet_w': 47.945495453647375,
   'x_dut_nfet_l': 61.4733460543701,
   'x_dut_cap_w': 42.63798102383294,
   'x_dut_cap_l': 47.5258072158167,
   'x_dut_res_s_l': 61.22858043459995,
   'x_dut_res_s_w': 46.80381736268717,
   'x_dut_res_3_l': 45.63395723124861,
   'x_dut_res_3_w': 45.875096046558674,
   '

In [42]:
circuit_optimizer.plot_loss(save_path=project_setup_yaml.parent / "loss_curve.html", show=True)

00:36:51 - SymXplorer.optimizer: [INFO] 📊 Plot saved to /foss/designs/eda/SymXplorer/examples/tunable-tia/ihp-sg13g2/spice/loss_curve.html
00:36:51 - SymXplorer.optimizer: [INFO] Opening interactive plot in browser...


## Inspection & Visualization

### (1) Best Param

In [43]:
out = circuit_optimizer.get_best_params()
if out is not None: 
    best_param, loss, metadata = out
metadata

00:36:51 - SymXplorer.optimizer: [INFO] best loss: 0.8061787273603156


{'fc': {'curr_val': np.float64(94553135.0),
  'loss': np.float64(0.2658038840861163)},
 'q': {'curr_val': np.float64(68.77241848320354),
  'loss': np.float64(5.877241848320354)},
 'gain_db': {'curr_val': np.float64(15.812604201891226),
  'loss': np.float64(0.5403748432741993)}}

In [44]:
# Print parameter sizes (convert to u)
for param in best_param:
    print(f"{param}: {best_param[param]*1e6 :0.2f}")

x_dut_nfet_w: 7.76
x_dut_nfet_l: 5.02
x_dut_cap_w: 50.99
x_dut_cap_l: 3547.93
x_dut_res_s_l: 851.24
x_dut_res_s_w: 619.61
x_dut_res_3_l: 418.63
x_dut_res_3_w: 407.99
vbias: 1688751.94


In [57]:
circuit_optimizer.plot_solution(best_param, show_plot=True, trace_name="vout")

00:37:26 - SymXplorer.optimizer: [INFO] total loss: 0.8061787273603156
00:37:26 - SymXplorer.optimizer: [INFO] 	Spec 'fc': curr_val=94553135.0, loss=0.2658038840861163
00:37:26 - SymXplorer.optimizer: [INFO] 	Spec 'q': curr_val=68.77241848320354, loss=5.877241848320354
00:37:26 - SymXplorer.optimizer: [INFO] 	Spec 'gain_db': curr_val=15.812604201891226, loss=0.5403748432741993


### (3) Metric Trace

In [46]:
circuit_optimizer.plot_optimization_trace(metric_x='fc', metric_y='gain_db', show=True)

00:36:52 - SymXplorer.optimizer: [INFO] Opening interactive plot in browser...


(tensor([7.6382e+06, 9.1354e+06, 9.3774e+06, 6.3641e+06, 8.3309e+06, 9.0035e+06,
         1.0305e+07, 8.9428e+06, 8.6916e+06, 9.3857e+06, 9.6965e+06, 8.2562e+06,
         1.1106e+07, 1.0397e+07, 8.1559e+06, 1.2154e+07, 8.6468e+06, 9.9149e+06,
         1.0275e+07, 1.1824e+07, 1.0375e+07, 9.5320e+06, 9.9639e+06, 1.0936e+07,
         1.0342e+07, 1.5261e+07, 1.2094e+07, 1.6864e+07, 9.0526e+06, 1.4628e+07,
         1.4338e+07, 1.3869e+07, 1.2321e+07, 9.8531e+06, 6.3759e+07, 1.0345e+07,
         1.7690e+07, 1.2132e+07, 1.5330e+07, 1.1647e+07, 1.7016e+07, 1.2004e+07,
         1.9313e+07, 1.1608e+07, 1.3970e+07, 3.6907e+07, 1.4426e+07, 1.6899e+07,
         1.5308e+07, 5.5539e+07, 1.2631e+07, 1.7910e+07, 1.6654e+07, 2.1692e+07,
         2.0172e+07, 1.4561e+07, 5.2729e+07, 3.9112e+07, 1.9623e+07, 1.2612e+07,
         1.3199e+07, 5.1488e+07, 3.1591e+07, 1.8512e+07, 2.5923e+07, 2.7779e+07,
         2.0753e+07, 2.0037e+07, 2.5924e+07, 2.3494e+07, 1.4211e+08, 2.6607e+07,
         4.5318e+07, 2.6730e

In [47]:
circuit_optimizer.plot_loss_value_by_spec(spec_name="gain_db", show=True)
circuit_optimizer.plot_loss_value_by_spec(spec_name="fc", show=True)

00:36:52 - SymXplorer.optimizer: [INFO] Opening interactive plot in browser...


00:36:52 - SymXplorer.optimizer: [INFO] Opening interactive plot in browser...


### (4) Design Space Exploration

In [48]:
circuit_optimizer.plot_design_space_exploration(param_x="x_dut_nfet_w", param_y="x_dut_nfet_l", show=True)
circuit_optimizer.plot_design_space_exploration(param_x="x_dut_cap_l", param_y="x_dut_cap_w", show=True)
circuit_optimizer.plot_design_space_exploration(param_x="vbias", param_y="x_dut_cap_w", show=True)

00:36:53 - SymXplorer.optimizer: [INFO] Opening interactive plot in browser...


00:36:53 - SymXplorer.optimizer: [INFO] Opening interactive plot in browser...


00:36:53 - SymXplorer.optimizer: [INFO] Opening interactive plot in browser...


(tensor([0.7343, 0.7039, 0.8600, 1.0719, 1.1207, 0.5986, 0.7964, 1.1958, 0.8989,
         1.1789, 1.1267, 1.0220, 1.0036, 1.0430, 0.9891, 1.1964, 1.0414, 1.1527,
         1.1874, 1.1779, 1.2330, 1.0740, 1.2053, 1.2744, 1.2372, 1.3960, 1.5415,
         1.1076, 1.4329, 1.0772, 1.3545, 1.1378, 1.3875, 1.0680, 1.3936, 0.9715,
         0.9488, 0.9771, 1.1944, 1.1871, 1.7840, 1.2827, 1.5279, 1.4480, 1.7324,
         1.7690, 1.7815, 1.5128, 1.7434, 1.5270, 1.3325, 1.5552, 1.4731, 1.5509,
         1.5164, 1.1898, 1.5514, 1.5349, 1.6798, 1.6585, 1.3249, 1.3110, 1.5427,
         1.2984, 1.4812, 1.2208, 1.3802, 1.5054, 1.3071, 1.7270, 1.2793, 1.5608,
         1.3070, 1.3502, 1.3004, 1.5174, 1.2717, 1.7066, 1.3631, 1.2885, 1.1936,
         1.4070, 1.4580, 1.5464, 1.5682, 1.6037, 1.6493, 1.3948, 1.7912, 1.4549,
         1.5744, 1.5583, 1.3997, 1.5516, 1.7391, 1.4737, 1.6129, 1.6888, 1.5295,
         1.4614, 1.7659, 1.5265, 1.7464, 1.7598, 1.5745, 1.5906, 1.6616, 1.7641,
         1.4894, 1.6173, 1.7

# Testing

## Loss Function Testing

In [49]:
# ----------------------------
# Instantiations
# ----------------------------
project_setup_yaml = Path(f"/foss/designs/eda/SymXplorer/examples/tunable-tia/ihp-sg13g2/spice/project_setup.yaml")
_ = setup_loggers()

# (1) Load the project setup information
PROJECT_SETUP = Project_Setup.from_yaml(project_setup_yaml)
PROJECT_SETUP

00:36:53 - SymXplorer: [INFO] 🚀 Logger initialized and ready!
00:36:53 - SymXplorer: [INFO] 📄 Log file: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/logs/SymXplorer_2025-10-02_00-36-53.log
00:36:53 - SymXplorer: [INFO] 🔧 spicelib logger set to 50
00:36:53 - SymXplorer.domains: [INFO] 📂 Loading project setup from /foss/designs/eda/SymXplorer/examples/tunable-tia/ihp-sg13g2/spice/project_setup.yaml
00:36:53 - SymXplorer.domains: [INFO] Initialized OptimizerConfig: CMA, type=nevergrad, budget=150, random_seed=48
00:36:53 - SymXplorer.domains: [INFO] 	Linear bounds: min=0, max=100
00:36:53 - SymXplorer.domains: [INFO] 	Log bounds: min=1, max=100
00:36:53 - SymXplorer.domains: [INFO] 	Loss function: max_loss=inf, norm_method=min-max, type=mse, rescale_mag=True, include_phase_loss=False, include_mag_loss=True
00:36:53 - SymXplorer.domains: [INFO] 	Number of target specs: 3
00:36:53 - SymXplorer.domains: [INFO] 		- TargetSpec(name=fc, target=100e6, range=1.00e+07 tolerance=500000

Project_Setup(name='Tunable-TIA', description='Tunable TIA BPF example sizing in the ihp-sg13g2 technology', simulator='ngspice', ws_root=PosixPath('/foss/designs/eda/SymXplorer'), netlist=PosixPath('examples/tunable-tia/ihp-sg13g2/spice/tb_ac.spice'), outdir=PosixPath('examples/tunable-tia/scripts/optimizer_output'), tech_spec=TechSpec(name='ihp-sg13g2', constraints={'max_nfet_w': np.float64(9.999999999999999e-06), 'min_nfet_w': np.float64(1.8e-07), 'max_nfet_l': np.float64(9.999999999999999e-06), 'min_nfet_l': np.float64(1.8e-07), 'max_pfet_w': np.float64(9.999999999999999e-06), 'min_pfet_w': np.float64(1.8e-07), 'max_pfet_l': np.float64(9.999999999999999e-06), 'min_pfet_l': np.float64(1.8e-07), 'max_cap_w': np.float64(0.01), 'min_cap_w': np.float64(1e-06), 'max_cap_l': np.float64(0.01), 'min_cap_l': np.float64(1e-06), 'max_res_w': np.float64(0.001), 'min_res_w': np.float64(1e-06), 'max_res_l': np.float64(0.001), 'min_res_l': np.float64(1e-06)}), pvt=PVT(temp=25, corner='tt', supply=

In [50]:
dummy_circuit_optimizer = Nevergrad_Spice_Multi_Spec_Constraint_Satisfaction(
    spicelib_wrapper=wrapper,
    setup_obj=PROJECT_SETUP
)

import numpy as np

gain_db_min  = np.float64(-135.0)
gain_db_max  = np.float64(135)
points_per_unit = 10
gain_db_vals = np.linspace(gain_db_min, gain_db_max, int((gain_db_max-gain_db_min) * (points_per_unit)))  

for gain_db in gain_db_vals:
    loss, fit_summary = dummy_circuit_optimizer.compute_fitness(performance_array={'gain_db' : gain_db})
    dummy_circuit_optimizer.optimization_log.append({
            "metric_value": None,
            "fit_summary": fit_summary,
            "params": None,
            "log": None
        })

dummy_circuit_optimizer.plot_loss_value_by_spec(spec_name='gain_db', show = True)

00:36:53 - SymXplorer.optimizer: [INFO] Initialized the Nevergrad_Spice_Multi_Spec_Optimizer with 3 target specs


00:36:55 - SymXplorer.optimizer: [INFO] Opening interactive plot in browser...


In [51]:
dummy_circuit_optimizer = Nevergrad_Spice_Multi_Spec_Constraint_Satisfaction(
    spicelib_wrapper=wrapper,
    setup_obj=PROJECT_SETUP
)

import numpy as np

fc_min  = 4
fc_max  = 10
points_per_unit = 1000

# num_points = int((fc_max-fc_min) * (points_per_unit/1e3))
logger.info(f"using {points_per_unit} points")
logger.info(f"\tTarget: {PROJECT_SETUP.optimizer_config.target_specs.get_target_by_name("fc")}")
# fc_vals = np.logspace(fc_min, fc_max)  
fc_vals = np.linspace(1e1, 1e8, 1000)  

for fc in fc_vals:
    loss, fit_summary = dummy_circuit_optimizer.compute_fitness(performance_array={'fc' : fc})
    dummy_circuit_optimizer.optimization_log.append({
            "metric_value": None,
            "fit_summary": fit_summary,
            "params": None,
            "log": None
        })

dummy_circuit_optimizer.plot_loss_value_by_spec(spec_name='fc', show = True)

00:36:55 - SymXplorer.optimizer: [INFO] Initialized the Nevergrad_Spice_Multi_Spec_Optimizer with 3 target specs
00:36:55 - SymXplorer.jupyter: [INFO] using 1000 points
00:36:55 - SymXplorer.jupyter: [INFO] 	Target: TargetSpec(name=fc, target=100e6, range=1.00e+07 tolerance=5000000.0, goal=exact, sim_type=ac, enable=True, error_type=relative-sigmoid, weight=1.0, enable=True, description=Center frequency)
00:36:56 - SymXplorer.optimizer: [INFO] Opening interactive plot in browser...


## Other

In [52]:
circuit_optimizer.optimization_log

[{'metric_value': np.float64(1.9246636224398537),
  'fit_summary': {'fc': {'curr_val': np.float64(7638218.0),
    'loss': np.float64(0.9998051203684322)},
   'q': {'curr_val': np.float64(8.624050455690952), 'loss': np.float64(0.0)},
   'gain_db': {'curr_val': np.float64(-24.864695269738707),
    'loss': np.float64(0.9248585020714215)}},
  'params': {'x_dut_nfet_w': 5.264026809450073e-06,
   'x_dut_nfet_l': 5.342080249320933e-06,
   'x_dut_cap_w': 0.005978527926150446,
   'x_dut_cap_l': 0.004857125655952536,
   'x_dut_res_s_l': 0.0003274609206267238,
   'x_dut_res_s_w': 0.00040511921153979616,
   'x_dut_res_3_l': 0.0004545325047515959,
   'x_dut_res_3_w': 0.00047612549679334576,
   'vbias': 0.734320065896732},
  'log': PosixPath('/foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/optimizer_output/run_1/tb_ac_1.log')},
 {'metric_value': np.float64(1.9086501343476847),
  'fit_summary': {'fc': {'curr_val': np.float64(9135382.5),
    'loss': np.float64(0.9997736499245804)},
   'q': {

In [53]:
PROJECT_SETUP.optimizer_config.target_specs.list_target_names()

['fc', 'q', 'gain_db']

In [54]:
target_spec = PROJECT_SETUP.optimizer_config.target_specs.get_target_by_name('gain_db')
target_spec

TargetSpec(name='gain_db', target=40, goal=<OptimizationGoalType.EXCEED: 'exceed'>, sim_type=<SimType.AC: 'ac'>, log_scale=False, enable=True, range=np.float64(20.0), error_type=<Error_Types.RELATIVE_SIGMOID: 'relative-sigmoid'>, weight=1.0, tolerance=5, description='gain in dB at fc')

In [55]:
circuit_optimizer.compute_spec_loss(curr_val=-90, target_spec=target_spec)

np.float64(0.996997635486526)

In [56]:
PROJECT_SETUP.dut_params

[Param(name='x_dut_nfet_w', min_val=np.float64(1.8e-07), max_val=np.float64(9.999999999999999e-06), val=None, description=None, log_scale=False),
 Param(name='x_dut_nfet_l', min_val=np.float64(1.8e-07), max_val=np.float64(9.999999999999999e-06), val=None, description=None, log_scale=False),
 Param(name='x_dut_cap_w', min_val=np.float64(1e-06), max_val=np.float64(0.01), val=None, description=None, log_scale=False),
 Param(name='x_dut_cap_l', min_val=np.float64(1e-06), max_val=np.float64(0.01), val=None, description=None, log_scale=False),
 Param(name='x_dut_res_s_l', min_val=np.float64(1e-06), max_val=np.float64(0.001), val=None, description=None, log_scale=False),
 Param(name='x_dut_res_s_w', min_val=np.float64(1e-06), max_val=np.float64(0.001), val=None, description=None, log_scale=False),
 Param(name='x_dut_res_3_l', min_val=np.float64(1e-06), max_val=np.float64(0.001), val=None, description=None, log_scale=False),
 Param(name='x_dut_res_3_w', min_val=np.float64(1e-06), max_val=np.fl